# 13 Canopy Ensemble


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

if Path('/kaggle/working').exists():
    ROOT = Path('/kaggle/working')
else:
    ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

ART = ROOT / 'artifacts'
MODEL_DIR = ART / 'models'
INF_DIR = ART / 'inference'
for p in [ART, MODEL_DIR, INF_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)


ROOT: C:\Users\alexy\Documents\ChatGPT_projects\LIDAR RORAIMA


In [2]:
from glob import glob
import joblib

feature_paths = sorted(glob(str(ROOT / 'artifacts' / 'features' / 'features_*_10m.parquet')))
if not feature_paths and Path('/kaggle/input').exists():
    feature_paths = sorted(str(p) for p in Path('/kaggle/input').rglob('features_*_10m.parquet'))
if not feature_paths:
    raise FileNotFoundError('No feature parquet found locally or in /kaggle/input')

features = pd.concat([pd.read_parquet(p) for p in feature_paths], ignore_index=True)
exclude = {'tile_id','zone_epsg','grid_x','grid_y','target_chm','target_landcover'}

model_files = []
for pattern in ['canopy_baseline.joblib','canopy_random_forest.joblib','canopy_boosting.joblib']:
    local = MODEL_DIR / pattern
    if local.exists():
        model_files.append(local)
    else:
        matches = list(Path('/kaggle/input').rglob(pattern)) if Path('/kaggle/input').exists() else []
        if matches:
            model_files.append(matches[0])

if len(model_files) < 1:
    raise FileNotFoundError(f'Need at least 1 canopy model file, found: {model_files}')

preds = []
for mf in model_files:
    payload = joblib.load(mf)
    cols = [c for c in payload['feature_columns'] if c in features.columns]
    X = features[cols].fillna(0)
    preds.append(payload['model'].predict(X))

stack = np.vstack(preds)
ensemble = features[['tile_id','grid_x','grid_y']].copy()
ensemble['pred_chm'] = stack.mean(axis=0)
ensemble['uncertainty_chm'] = stack.std(axis=0)
out_path = INF_DIR / 'canopy_ensemble.parquet'
ensemble.to_parquet(out_path, index=False)
out_path


WindowsPath('C:/Users/alexy/Documents/ChatGPT_projects/LIDAR RORAIMA/artifacts/inference/canopy_ensemble.parquet')

In [3]:
out_csv = MODEL_DIR / 'model_results.csv'
pd.read_csv(out_csv).tail(10) if out_csv.exists() else pd.DataFrame()


,timestamp_utc,project,model_name,params_hash,fold_scheme,artifact_path,kaggle_notebook_url,mae,r2,rmse,macro_f1,macro_f1_recomputed,macro_precision,macro_recall
0,2026-03-03T16:42:38.484045+00:00,canopy,baseline,7d54bdb74401,tile_blocked_5,C:\Users\alexy\Documents\ChatGPT_projects\LIDA...,NaN,1.559589,-0.042135,2.123754,NaN,NaN,NaN,NaN
1,2026-03-03T16:42:43.128144+00:00,landcover,baseline,1c31f279558f,tile_blocked_5,C:\Users\alexy\Documents\ChatGPT_projects\LIDA...,NaN,NaN,NaN,NaN,0.293425,0.293425,0.262818,0.333333
2,2026-03-03T16:53:11.522798+00:00,canopy,baseline,7d54bdb74401,tile_blocked_5,C:\Users\alexy\Documents\ChatGPT_projects\LIDA...,NaN,1.559589,-0.042135,2.123754,NaN,NaN,NaN,NaN
3,2026-03-03T16:53:13.253933+00:00,landcover,baseline,1c31f279558f,tile_blocked_5,C:\Users\alexy\Documents\ChatGPT_projects\LIDA...,NaN,NaN,NaN,NaN,0.293425,0.293425,0.262818,0.333333
4,2026-03-05T17:57:02.595880+00:00,landcover,baseline,1c31f279558f,tile_blocked_5,C:\Users\alexy\Documents\ChatGPT_projects\LIDA...,NaN,NaN,NaN,NaN,0.293425,0.293425,0.262818,0.333333
5,2026-03-05T18:46:40.092773+00:00,landcover,random_forest,beadd2050b20,tile_blocked_5,C:\Users\alexy\Documents\ChatGPT_projects\LIDA...,NaN,NaN,NaN,NaN,0.431744,0.431744,0.498021,0.411500
6,2026-03-05T19:07:12.912023+00:00,landcover,boosting,71f4f49d4a60,tile_blocked_5,C:\Users\alexy\Documents\ChatGPT_projects\LIDA...,NaN,NaN,NaN,NaN,0.443114,0.443114,0.492892,0.437373
